<center><h1>Netflix Prize</h1></center>

In [1]:
from src import *
import src
import numpy as np
from importlib import reload
import pandas as pd
reload(src)
import matplotlib.pyplot as plt
from math import sqrt

import warnings
warnings.filterwarnings('ignore')

## Pré-processamento

In [22]:
def df_to_latex(df, legenda):
    latex = r'''\begin{table}[]
                \centering
                        '''
    latex = latex + df.to_latex(index=False, header=True, 
                                column_format='c'*len(df.columns.to_list()))
    latex = latex + r'''
                    \caption{%s}
                    \end{table}
                        ''' % legenda 
    return latex

### Carregamento

In [2]:
n = pd.read_csv('data/raw/netflix/movie_titles.csv', 
                names = ['movieId', 'year', 'name'], 
                encoding='ISO-8859-1',
               lineterminator='\n',
               usecols=range(3))[['movieId', 'name']]

In [3]:
n

,movieId,name
0,1,Dinosaur Planet
1,2,Isle of Man TT 2004 Review
2,3,Character
3,4,Paula Abdul's Get Up & Dance
4,5,The Rise and Fall of ECW
...,...,...
17765,17766,Where the Wild Things Are and Other Maurice Se...
17766,17767,Fidel Castro: American Experience
17767,17768,Epoch
17768,17769,The Company


In [4]:
n_g = pd.read_csv('data/raw/netflix/netflix_genres.csv')

In [21]:
n_g

,movieId,genres
0,1,Documentary|Animation|Family
1,3,Crime|Drama|Mystery
2,4,Family
3,5,Documentary|Sport
4,6,Documentary
...,...,...
12274,17764,Comedy|Drama|History|Romance
12275,17765,Action|Adventure|Family|Sci-Fi
12276,17768,Action|Drama|Fantasy
12277,17769,Drama|Music|Romance


In [5]:
n_g_found = n_g['movieId'].to_list()

In [6]:
n[~n['movieId'].isin(n_g_found)]

,movieId,name
1,2,Isle of Man TT 2004 Review
9,10,Fighter
10,11,Full Frame: Documentary Shorts
12,13,Lord of the Rings: The Return of the King: Ext...
13,14,Nature: Antarctica
...,...,...
17749,17750,The Hee Haw Collection: Vol. 4
17750,17751,Highlander: Season 2
17756,17757,Ulysses S. Grant: Warrior / President: America...
17765,17766,Where the Wild Things Are and Other Maurice Se...


In [28]:
n_g['g'] = n_g['genres'].str.split('|')

In [31]:
n_g.explode('g')['g'].nunique()

27

In [23]:
print(df_to_latex(n_g.head(), 'Primeiras 5 linhas do conjunto de metadados de filmes presentes no conjunto de dados Netflix Prize com os gêneros obtidos pelo cruzamento com o conjunto IMDb'))

\begin{table}[]
                \centering
                        \begin{tabular}{cc}
\toprule
movieId & genres \\
\midrule
1 & Documentary|Animation|Family \\
3 & Crime|Drama|Mystery \\
4 & Family \\
5 & Documentary|Sport \\
6 & Documentary \\
\bottomrule
\end{tabular}

                    \caption{Primeiras 5 linhas do conjunto de metadados de filmes presentes no conjunto de dados Netflix Prize com os gêneros obtidos pelo cruzamento com o conjunto IMDb}
                    \end{table}
                        


## Experimentos

In [2]:
userColumn='userId'
itemColumn='artistId'
ratingColumn='rating'

In [3]:
loader = DataLoader(file_path='data/processed/yahoo_music/yahoo_music_tratado.csv',
                    separator=',',
                    header=['userId', 'artistId', 'rating'],
                    genre_data_path='data/processed/yahoo_music/yahoo_music_artists_metadata_with_genres.csv',
                    genre_data_separator=',',
                    genre_data_header=['artistId', 'artistName', 'genres'],
                    genre_item_column='artistId',
                    genre_genre_column='genres',
                    genre_genre_separator='|',
                    genre_data_encoding='ISO-8859-1'
                    )

24/09/06 21:24:53 WARN Utils: Your hostname, david-Nitro resolves to a loopback address: 127.0.1.1; using 192.168.15.14 instead (on interface wlp9s0)
24/09/06 21:24:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/09/06 21:24:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = loader.load_data()
df_pandas = loader.convert_to_pandas(df)
genres_map = loader.get_genre_map()
# (train, test) = loader.train_test_separator(df)

In [5]:
df_pandas

,userId,artistId,rating
0,12722,1000276,5.0
1,12722,1000335,5.0
2,12722,1000418,5.0
3,12722,1000454,5.0
4,12722,1000656,5.0
...,...,...,...
112518,1942234,1097451,5.0
112519,1942234,1098966,5.0
112520,1942234,1099341,5.0
112521,1942234,1099448,5.0


## ALS

In [6]:
kfold = KFOLD(df_pandas, 'FC', userColumn, itemColumn, ratingColumn, genres_map)

24/09/06 21:25:00 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [ ]:
results = kfold.run_cross_validation()

1) Running FC
******************************
Lambda = 0.0
Reseting metrics lists
********************
FAIRNESS MEASURE: kullback-leibler
=====1º Fold=====
Train/test spliting
Instantiating ALS...


24/09/06 21:25:05 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
24/09/06 21:25:08 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
24/09/06 21:25:11 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Getting n = 100 predictions for each user


Calibrating...
CALCULATING MRMC
MRMC = 0.3252801252573285
CALCULATING MACE
MACE = 0.001991729290430589
CALCULATING MAP
MAP = 0.008682745825602968
CALCULATING MRR
MRR = 0.008682745825602968
=====2º Fold=====
Train/test spliting
Instantiating ALS...
Getting n = 100 predictions for each user
Calibrating...
CALCULATING MRMC
MRMC = 0.3712320246142023
CALCULATING MACE
MACE = 0.0021874243253553606
CALCULATING MAP
MAP = 0.003116883116883117
CALCULATING MRR
MRR = 0.003116883116883117
=====3º Fold=====
Train/test spliting
Instantiating ALS...
Getting n = 100 predictions for each user
Calibrating...
CALCULATING MRMC
MRMC = 0.29430458612782917
CALCULATING MACE
MACE = 0.0013401230803828216
CALCULATING MAP
MAP = 0.0014285714285714288
CALCULATING MRR
MRR = 0.0014285714285714288
=====4º Fold=====
Train/test spliting
Instantiating ALS...
Getting n = 100 predictions for each user
Calibrating...
CALCULATING MRMC
MRMC = 0.3751154443408931
CALCULATING MACE
MACE = 0.0022264550264550266
CALCULATING MAP
MAP =

Calibrating...
CALCULATING MRMC
MRMC = 0.36147048798950404
CALCULATING MACE
MACE = 0.001991729290430589
CALCULATING MAP
MAP = 0.008682745825602968
CALCULATING MRR
MRR = 0.008682745825602968
=====2º Fold=====
Train/test spliting
Instantiating ALS...
Getting n = 100 predictions for each user
Calibrating...
CALCULATING MRMC
MRMC = 0.3939398400243541
CALCULATING MACE
MACE = 0.0021874243253553606
CALCULATING MAP
MAP = 0.003116883116883117
CALCULATING MRR
MRR = 0.003116883116883117
=====3º Fold=====
Train/test spliting
Instantiating ALS...
Getting n = 100 predictions for each user
Calibrating...
CALCULATING MRMC
MRMC = 0.2655579094505297
CALCULATING MACE
MACE = 0.0013401230803828216
CALCULATING MAP
MAP = 0.0014285714285714288
CALCULATING MRR
MRR = 0.0014285714285714288
=====4º Fold=====
Train/test spliting
Instantiating ALS...
Getting n = 100 predictions for each user
Calibrating...
CALCULATING MRMC
MRMC = 0.42231065541011453
CALCULATING MACE
MACE = 0.0022264550264550266
CALCULATING MAP
MAP 

Calibrating...
CALCULATING MRMC
MRMC = 0.11332756633825354
CALCULATING MACE
MACE = 0.0012193520764949323
CALCULATING MAP
MAP = 0.004862914862914863
CALCULATING MRR
MRR = 0.004862914862914863
=====2º Fold=====
Train/test spliting
Instantiating ALS...
Getting n = 100 predictions for each user


Calibrating...
CALCULATING MRMC
MRMC = 0.12509547334614296
CALCULATING MACE
MACE = 0.0010055122764482361
CALCULATING MAP
MAP = 0.0028571428571428576
CALCULATING MRR
MRR = 0.0028571428571428576
=====3º Fold=====
Train/test spliting
Instantiating ALS...
Getting n = 100 predictions for each user
Calibrating...
CALCULATING MRMC
MRMC = 0.1273634778879431
CALCULATING MACE
MACE = 0.000703240962981223
CALCULATING MAP
MAP = 0.0011327561327561329
CALCULATING MRR
MRR = 0.0011327561327561329
=====4º Fold=====
Train/test spliting
Instantiating ALS...
Getting n = 100 predictions for each user
Calibrating...
CALCULATING MRMC
MRMC = 0.12066198195781472
CALCULATING MACE
MACE = 0.0007875394446823017
CALCULATING MAP
MAP = 0.003896103896103896
CALCULATING MRR
MRR = 0.0047619047619047615
=====5º Fold=====
Train/test spliting
Instantiating ALS...
Getting n = 100 predictions for each user
Calibrating...
CALCULATING MRMC
MRMC = 0.11481539217771888
CALCULATING MACE
MACE = 0.0009040050858232669
CALCULATING MAP


Calibrating...
CALCULATING MRMC
MRMC = 0.42049545075045364
CALCULATING MACE
MACE = 0.0009944024229738507
CALCULATING MAP
MAP = 0.003823953823953824
CALCULATING MRR
MRR = 0.003823953823953824
=====2º Fold=====
Train/test spliting
Instantiating ALS...
Getting n = 100 predictions for each user
Calibrating...
CALCULATING MRMC
MRMC = 0.42101093642414256
CALCULATING MACE
MACE = 0.0005825434952170288
CALCULATING MAP
MAP = 0.003463203463203463
CALCULATING MRR
MRR = 0.003463203463203463
=====3º Fold=====
Train/test spliting
Instantiating ALS...
Getting n = 100 predictions for each user
Calibrating...
CALCULATING MRMC
MRMC = 0.46350693268707577
CALCULATING MACE
MACE = 0.0006917844060701205
CALCULATING MAP
MAP = 0.0019109461966604826
CALCULATING MRR
MRR = 0.0019109461966604826
=====4º Fold=====
Train/test spliting
Instantiating ALS...
Getting n = 100 predictions for each user
Calibrating...
CALCULATING MRMC
MRMC = 0.41779675831286955
CALCULATING MACE
MACE = 0.0006131363585909037
CALCULATING MAP
M

In [ ]:
results

In [ ]:
results.to_csv('results/yahoo_movies/results_yahoo_movies2.csv', index=False)

In [ ]:
# results = pd.read_csv('results/yahoo_movies/results_yahoo_movies2.csv')

In [ ]:
#results.to_csv('results/yahoo_movies/results_yahoo_movies.csv', index=False)

In [ ]:
tradeoff_weights = results['Lambda'].drop_duplicates().to_list()
mrmc_kl = results[(results['FairnessMeasure'] == 'kullback-leibler') & (results['Statistic'] == 'mean')]['MAP'].to_list()
mrmc_he = results[(results['FairnessMeasure'] == 'hellinger') & (results['Statistic'] == 'mean')]['MAP'].to_list()
mrmc_pe = results[(results['FairnessMeasure'] == 'pearson') & (results['Statistic'] == 'mean')]['MAP'].to_list()

# Plotando o gráfico
plt.figure(figsize=(10, 6))
plt.plot(tradeoff_weights, mrmc_kl, marker='o', linestyle='-', color='b', label='Kullback-Leibler')
plt.plot(tradeoff_weights, mrmc_he, marker='s', linestyle='-', color='r', label='Hellinger')
plt.plot(tradeoff_weights, mrmc_pe, marker='^', linestyle='-', color='g', label='Pearson chi-quadrado')

plt.xlabel('λ')
plt.xticks(np.arange(0.0, 1.1, 0.1))
plt.ylabel('Mean rank miscalibration (MRMC)')
plt.title('MovieLens com ALS')
plt.legend()
plt.grid(True)
plt.show()


#### CRIANDO GRÁFICOS

In [ ]:
def show_graphic(dataset_name, metric, title):
    short_metric = metric[metric.find('(')+1: metric.find(')')]
    tradeoff_weights = results['Lambda'].drop_duplicates().to_list()
    mrmc_kl = results[(results['FairnessMeasure'] == 'kullback-leibler') & (results['Statistic'] == 'mean')][short_metric].to_list()
    mrmc_he = results[(results['FairnessMeasure'] == 'hellinger') & (results['Statistic'] == 'mean')][short_metric].to_list()
    mrmc_pe = results[(results['FairnessMeasure'] == 'pearson') & (results['Statistic'] == 'mean')][short_metric].to_list()

    mrmc_he_err = results[(results['FairnessMeasure'] == 'hellinger') & (results['Statistic'] == 'stdev')][short_metric].to_list()
    mrmc_he_err = [std/sqrt(5) for std in mrmc_he_err]
    mrmc_kl_err = results[(results['FairnessMeasure'] == 'kullback-leibler') & (results['Statistic'] == 'stdev')][short_metric].to_list()
    mrmc_kl_err = [std/sqrt(5) for std in mrmc_kl_err]
    mrmc_pe_err = results[(results['FairnessMeasure'] == 'pearson') & (results['Statistic'] == 'stdev')][short_metric].to_list()
    mrmc_pe_err = [std/sqrt(5) for std in mrmc_pe_err]
    
    # Plotando o gráfico com barras de erro
    plt.figure(figsize=(10, 6))
    plt.errorbar(tradeoff_weights, mrmc_kl, yerr=mrmc_kl_err, marker='o', linestyle='-', color='b', label='Kullback-Leibler', capsize=5) 
    plt.errorbar(tradeoff_weights, mrmc_he, yerr=mrmc_he_err, marker='s', linestyle='-', color='r', label='Hellinger', capsize=5) 
    plt.errorbar(tradeoff_weights, mrmc_pe, yerr=mrmc_pe_err, marker='^', linestyle='-', color='g', label='Pearson chi-quadrado', capsize=5)
    
    # Configurando os rótulos e título
    plt.xlabel(r'$\lambda$')
    plt.ylabel(metric)
    plt.title(title)
    
    # Definindo os valores do eixo x para incrementar de 0.1 em 0.1
    plt.xticks(np.arange(0.0, 1.1, 0.1))
    
    plt.legend()
    plt.grid(True)
    #plt.savefig(f'results/{dataset_name}/graphs/{dataset_name}_{metric}.pgf'.replace(' ', '_'))
    plt.savefig(f'results/{dataset_name}/graphs/{dataset_name}_{metric}.pdf'.replace(' ', '_'), bbox_inches='tight')
    plt.show()
    

In [ ]:
for metric in ['Mean Average Calibration Error (MACE)', 'Mean Rank Miscalibration (MRMC)', 'Mean Average Precision (MAP)', 'Mean Reciprocal Rank (MRR)']:
    print('=' * 10 + metric + '=' * 10)
    show_graphic('yahoo_movies', metric, 'Yahoo Movies com ALS')



In [ ]:
predictions_user_map

In [ ]:
distributions = Distributions(train.toPandas(), userColumn, itemColumn, genres_map)

In [ ]:
distributions.show_userProfileDist_userRecDist(predictions_user_map, 305)


# FBC-kNN

In [ ]:
df_genres = loader.genre_data

In [ ]:
df_genres

In [ ]:
df_expanded = df_genres['genres'].str.get_dummies(sep='|')

In [ ]:
df_final = pd.concat([df_genres[['item']], df_expanded], axis=1)

In [ ]:
df_final.index = df_final['item']

In [ ]:
df_mv_genres = df_final.drop(columns = ['item'])

In [ ]:
df_mv_genres

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
cosine_sim = cosine_similarity(df_mv_genres)

In [ ]:
df_cosine_sim = pd.DataFrame(cosine_sim, index = df_mv_genres.index, columns = df_mv_genres.index)

In [ ]:
df_cosine_sim

In [ ]:
similar_itens = list(df_cosine_sim.loc[1].sort_values(ascending=False).keys())[:102]
similar_itens.remove(1)
#rated_by_u = set(df_pandas[(df_pandas[userColumn] == 1) & (df_pandas[ratingColumn] > 0)][itemColumn].to_list())
#intersect = list(similar_itens.intersection(rated_by_u))[:101]


In [ ]:
def get_recommendatinon(movieId, userId, df_sim, df_ratings, k):
    similar_itens = list(df_sim.loc[movieId].sort_values(ascending=False).keys())[:k+2]
    similar_itens.remove(movieId)
    return similar_itens()


In [ ]:
#def get_recommendatinon(movieId, userId, df_sim, df_ratings, k):
#    similar_itens = set(df_sim.loc[movieId].sort_values(ascending=False).keys())
#    rated_by_u = set(df_ratings[(df_ratings[userColumn] == userId) & (df_ratings[ratingColumn] > 0)][itemColumn].to_list())
#    intersect = list(similar_itens.intersect(rated_by_u))[:k+1]
